# N11 Tape Architecture Sweep

Runs the tape architecture comparison using the core settings from `2d_tape_ICNN.ipynb`. Outputs include per-run artifacts, paper-ready architecture comparison plots, regenerated energy landscapes, and post-training Hessian diagnostics.

In [ ]:
import os
from pathlib import Path

import jax
import jax.numpy as jnp

jax.config.update("jax_enable_x64", True)
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

from properties import TapeN11Properties
from run_architectures import SweepConfig

ROOT = Path.cwd()
train_file = "../experiment_data/tape_data/11_noded/n11_tape_train_dataset.npz"
valid_file = "../experiment_data/tape_data/11_noded/n11_tape_test_dataset.npz"
properties = TapeN11Properties(mass=-0.005)

# Copied from 2d_tape_ICNN.ipynb
K_init_chol = (0.2, 0.0, 0.1)
K_init_diag = (0.2, 0.1)

base_cfg = SweepConfig(
    der_K_diag=K_init_diag,
    der_K_chol=K_init_chol,
    hidden=(10,),
    corr_factor=0.01,
    input_mode="raw",
    only_stretching_NN=False,
    only_bending_NN=False,
    zero_reference=True,
    activation="tanh",
    mode="anisotropic",
    n_epochs=1000,
    lr=5e-2,
    weight_decay=1e-5,
    seed=42,
    valid_every=10,
    max_dlambda=5e-2,
    iters=10,
    ls_steps=10,
    abs_tol=5e-4,
    rel_tol=1e-4,
    early_stop=True,
    train_fail_on_nonconvergence=True,
    prediction_fail_on_nonconvergence=False,
    hessian_reg_strength=1e-6,
    hessian_reg_probes=1,
    hessian_reg_seed=0,
    force_key=None,
    force_loss_strength=0.0,
    force_components=(0, 1, 2),
    force_sign=1.0,
    return_loss_components=False,
    early_stopping=True,
    early_stopping_patience=200,
    early_stopping_min_delta=1e-5,
    restore_best_model=True,
    save_npz=True,
    save_model=True,
    save_plots=True,
    save_force_predictions=False,
    plot_force_predictions=False,
    save_hessian_diagnostics=False,
    save_energy_landscapes=False,
    verbose=True,
    continue_on_failure=True,
)

print(properties)

OUTPUT_DIR = ROOT / "arch_sweep_outputs_n11_tape_all_architectures"
PAPER_PLOT_DIR = OUTPUT_DIR / "paper_ready_architecture_comparison"
base_cfg = base_cfg.__class__(**{**base_cfg.__dict__, "output_dir": str(OUTPUT_DIR)})
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Saving architecture sweep results under: {OUTPUT_DIR.resolve()}")


In [ ]:
from run_architectures import subset_tape_tube_candidates, subset_brazier_stiffness_only

selected_architectures = subset_tape_tube_candidates()

# For a quicker debug pass, use only the models closest to 2d_tape_ICNN.ipynb:
# selected_architectures = [
#     "brazier_chol_stiffness_baseline",
#     "brazier_chol_stiffness_mlp",
#     "brazier_chol_stiffness_icnn",
# ]

print(f"Running {len(selected_architectures)} architectures:")
for name in selected_architectures:
    print(f"  - {name}")
print("\nPer-architecture plots:", base_cfg.save_plots)
print("Energy landscape snapshots during training:", base_cfg.save_energy_landscapes)


In [ ]:
from run_architectures import run_architecture_sweep

results = run_architecture_sweep(
    properties=properties,
    train_file=train_file,
    valid_file=valid_file,
    cfg=base_cfg,
    selected_architectures=selected_architectures,
)

successes = [name for name, result in results.items() if result["success"]]
failures = {name: result["failure_reason"] for name, result in results.items() if not result["success"]}

print(f"Succeeded: {len(successes)}/{len(results)}")
if failures:
    print("Failures:")
    for name, reason in failures.items():
        print(f"  - {name}: {reason}")
else:
    print("No architecture failures.")


In [ ]:
from run_architectures import generate_energy_landscapes_for_sweep

energy_landscape_paths = generate_energy_landscapes_for_sweep(
    str(OUTPUT_DIR),
    architectures=successes if "successes" in globals() else selected_architectures,
    include_initial=False,
    include_final=True,
    use_valid=True,
    traj_idx=0,
    dpi=180,
    n_grid=None,
    continue_on_failure=True,
)
print("Energy landscape plots regenerated for", len(energy_landscape_paths), "runs.")


In [ ]:
from architecture_plots import plot_architecture_comparison_paper, plot_summary_final_losses

PAPER_PLOT_DIR.mkdir(parents=True, exist_ok=True)
paper_architectures = successes if "successes" in globals() else selected_architectures
paper_paths = plot_architecture_comparison_paper(
    architectures=paper_architectures,
    results_dir=str(OUTPUT_DIR),
    output_dir=str(PAPER_PLOT_DIR),
    traj_idx=0,
)
plot_summary_final_losses(
    {name: results[name] for name in paper_architectures},
    save_path=str(PAPER_PLOT_DIR / "final_loss_summary.png"),
    show=False,
)
print("Paper-ready plots written to:", PAPER_PLOT_DIR.resolve())
for key, path in paper_paths.items():
    if key != "colors":
        print(f"  {key}: {path}")


In [ ]:
import subprocess
import sys

HESSIAN_USE_PREDICTED = True
HESSIAN_STRIDE = 10
HESSIAN_MAX_TRAJECTORIES = 1
HESSIAN_SPLITS = ("train", "valid")

cmd = [
    sys.executable,
    "compute_architecture_hessian_diagnostics.py",
    str(OUTPUT_DIR),
    "--stride", str(HESSIAN_STRIDE),
    "--splits", *HESSIAN_SPLITS,
]
if HESSIAN_USE_PREDICTED:
    cmd.append("--use-predicted")
if HESSIAN_MAX_TRAJECTORIES is None:
    cmd.append("--all-trajectories")
else:
    cmd.extend(["--max-trajectories", str(HESSIAN_MAX_TRAJECTORIES)])

print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=str(ROOT), check=True)
print("Hessian diagnostics table:", OUTPUT_DIR / "hessian_diagnostics_table.csv")


In [ ]:
print("Done. Key outputs:")
print("  - <architecture>/results.npz")
print("  - <architecture>/model.eqx")
print("  - <architecture>/hessian_diagnostics_summary.json")
print("  - hessian_diagnostics_table.csv")
print("  - paper_ready_architecture_comparison/*.pdf")
